In [0]:
from pyspark.sql import functions as F
from delta.tables import DeltaTable

In [0]:
%run /Workspace/consolidated-pipeline/1st_setup/Utilities

In [0]:
print(silver_schema)

In [0]:
dbutils.widgets.text('Catalog','fmcg','catalog')
dbutils.widgets.text('Data_Source','orders','data_source')
catalog = dbutils.widgets.get('Catalog')
data_source = dbutils.widgets.get('Data_Source')

base_path = f"s3://amzn-s3-sportsbar/{data_source}"
landing_path = f"{base_path}/landing/"
processed_path = f"{base_path}/processed/"

print(base_path)
print(landing_path)
print(processed_path)


In [0]:
# define the tables
bronze_table = f"{catalog}.{bronze_schema}.{data_source}"
silver_table = f"{catalog}.{silver_schema}.{data_source}"
gold_table = f"{catalog}.{gold_schema}.sb_fact_{data_source}"

In [0]:
df = (
    spark.read.format('csv')
    .option("header",True)
    .option("infer_schema",True)
    .csv(f'{landing_path}/*.csv')
    .withColumn('readTimeStamp',F.current_timestamp())
    .select('*','_metadata.file_name','_metadata.file_size')
)
print("Total Rows :", df.count())
df.show(5)

In [0]:
display(df.limit(20))

In [0]:
df.write\
    .format("delta")\
    .option("delta.enableChangeDataFeed",True)\
    .mode("append")\
    .saveAsTable(bronze_table)

In [0]:
files = dbutils.fs.ls(landing_path)

for file_info in files:
    dbutils.fs.mv(
        file_info.path,
        f"{processed_path}/{file_info.name}",
        True
    )

# Silver Processing

In [0]:
df_order = spark.sql(f"Select * from {bronze_table}")
df_order.show(2)

In [0]:
display(df_order.limit(20))

In [0]:
df_order.printSchema()

### Transfromations

In [0]:
# Not Null Values
df_order = df_order.filter(F.col("order_qty").isNotNull())

# Clean customer id, else set to have 99999
df_order = df_order.withColumn(
  "customer_id",
  F.when(F.col("customer_id").rlike("^[0-9]+$") , F.col("customer_id"))
  .otherwise("99999")
  .cast("string")
)

# Removing weekday name from order_placement_date 
#  "Tuesday, July 01, 2025" → "July 01, 2025"

df_order = df_order.withColumn(
  "order_placement_date",
  F.regexp_replace(
    F.col("order_placement_date"),
    r"^[A-Za-z]+,\s*",
    ""
  )
)

# Parse order_placement_date using multiple possible formats
df_order = df_order.withColumn(
  "order_placement_date",
  F.coalesce(
    F.try_to_date("order_placement_date","yyyy/MM/dd"),
    F.try_to_date("order_placement_date","dd-MM-yyyy"),
    F.try_to_date("order_placement_date","dd/MM/yyyy"),
    F.try_to_date("order_placement_date","MMMM dd, yyyy")
  )
)

# Drop Duplicates
df_order = df_order.dropDuplicates(["order_id", "order_placement_date", "customer_id", "product_id", "order_qty"])


In [0]:
display(df_order.limit(20))

In [0]:
# checking the maximum and minimum date
df_order.agg(
    F.min("order_placement_date").alias("min_date"),
    F.max("order_placement_date").alias("max_date")
).show()

In [0]:
# Join in products
df_products = spark.table("fmcg.silver.products")
df_join = df_order.join(df_products,
                        on='product_id',
                        how='inner')\
                .select(df_order['*'],df_products['product_code'])


In [0]:
display(df_join.limit(10))

In [0]:
if not (spark.catalog.tableExists(silver_table)):
    df_join.write.format('delta')\
    .option('delta.enableChangeDataFeed',True)\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable(silver_table)
else :
    silver_delta = DeltaTable.forName(spark,silver_table)
    silver_delta.alias('silver').merge(df_join.alias('bronze'),
    "silver.order_placement_date = bronze.order_placement_date AND silver.order_id = bronze.order_id AND silver.product_code = bronze.product_code AND silver.customer_id = bronze.customer_id").whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# Gold Processing

In [0]:
df_gold = spark.sql(f"SELECT order_id, order_placement_date as date, customer_id as customer_code,product_code, product_id, order_qty as sold_quantity  from {silver_table}")
df_gold.show(2)

In [0]:
gold_table

In [0]:
if not (spark.catalog.tableExists(gold_table)) :
    print("Creating Gold Table")
    df_gold.write\
    .format('delta')\
    .option('delta.enableChangeDataFeed',True)\
    .option('mergeSchema',True)\
    .mode('overwrite')\
    .saveAsTable(gold_table)
else :
    gold_delta = DeltaTable.forName(spark,gold_table)
    gold_delta.alias('source')\
     .merge(df_gold.alias('gold'),
            'source.date= gold.date AND source.order_id = gold.order_id AND source.product_code = gold.product_code AND source.customer_code = gold.customer_code')\
        .whenMatchedUpdateAll()\
        .whenNotMatchedInsertAll()\
        .execute()

# merging data with parent

In [0]:
df_child = spark.sql(f"Select date, product_code,customer_code, sold_quantity from {gold_table}")
df_child.show(2)

In [0]:
df_child.count()

In [0]:
df_monthly = (
    df_child.withColumn("month_start", F.trunc('date','MM'))
    .groupBy("month_start","product_code","customer_code")
    .agg(F.sum("sold_quantity").alias("sold_quantity"))
    .withColumnRenamed("month_start","date")
)

df_monthly.show(5,truncate =False)

In [0]:
df_monthly.count()

In [0]:
gold_parent_delta = DeltaTable.forName(spark, f"{catalog}.{gold_schema}.fact_orders")
gold_parent_delta.alias("parent_gold")\
    .merge(df_monthly.alias("child_gold"),
      "parent_gold.date = child_gold.date AND parent_gold.product_code = child_gold.product_code AND parent_gold.customer_code = child_gold.customer_code"
        ).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()